주요 출력:
- single-output LightGBM의 test 성능 확인
- single-output within t+2 기준 permutation feature importance
- single-output LightGBM built-in gain importance
- calibration, decision curve analysis(DCA)
- `RUN_SHAP = True`일 때 Tree SHAP feature importance와 beeswarm plot


In [ ]:
from pathlib import Path
import json
import random
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

warnings.filterwarnings("ignore")


In [ ]:
# 작업 위치
def find_project_dir() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (
            (candidate / "processed" / "data_split").exists()
            and (candidate / "models").exists()
            and (candidate / "outputs").exists()
        ):
            return candidate
        parkinson_dir = candidate / "Parkinson"
        if (
            (parkinson_dir / "processed" / "data_split").exists()
            and (parkinson_dir / "models").exists()
            and (parkinson_dir / "outputs").exists()
        ):
            return parkinson_dir
    raise FileNotFoundError("Could not locate the Parkinson project directory from the current working directory.")


PROJECT_DIR = find_project_dir()
DATA_SPLIT_DIR = PROJECT_DIR / "processed" / "data_split"
CLEAN_DATA_DIR = PROJECT_DIR / "models" / "clean_data"
MODEL_DIR = PROJECT_DIR / "models"
MODELING_OUTPUT_DIR = PROJECT_DIR / "outputs" / "modeling"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "model_interpretation"
FIGURE_DIR = OUTPUT_DIR / "figures"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_SPLIT_DIR:", DATA_SPLIT_DIR)
print("MODEL_DIR:", MODEL_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
# 설정값 (config)
RANDOM_STATE = 42
TARGET_WINDOW_SIZE = 3

SINGLE_OUTPUT_MODEL_PATH = MODEL_DIR / "within_t_plus_2" / "lgbm_t_point_within_t_plus_2.joblib"
FEATURE_COLUMNS_PATH = CLEAN_DATA_DIR / "lstm_feature_columns.json"

EXPLAIN_SAMPLE_SIZE = 512
PERMUTATION_REPEATS = 3
TOP_N_PLOT = 25

CALIBRATION_N_BINS = 10
DCA_THRESHOLDS = np.round(np.arange(0.01, 0.81, 0.01), 2)

RUN_SHAP = True
SHAP_EXPLAIN_SIZE = 512


In [ ]:
# seed 설정
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
rng = np.random.default_rng(RANDOM_STATE)


## 전처리 산출물 로딩


In [ ]:
required_files = {
    "X_train": DATA_SPLIT_DIR / "X_train_lstm.npy",
    "X_test": DATA_SPLIT_DIR / "X_test_lstm.npy",
    "y_train_steps": DATA_SPLIT_DIR / "y_train_steps_lstm.npy",
    "y_test_steps": DATA_SPLIT_DIR / "y_test_steps_lstm.npy",
    "y_train_step_mask": DATA_SPLIT_DIR / "y_train_step_mask_lstm.npy",
    "y_test_step_mask": DATA_SPLIT_DIR / "y_test_step_mask_lstm.npy",
    "meta_train": DATA_SPLIT_DIR / "lstm_train_metadata.csv",
    "meta_test": DATA_SPLIT_DIR / "lstm_test_metadata.csv",
    "preprocessor": CLEAN_DATA_DIR / "lstm_preprocessor.joblib",
    "single_output_model": SINGLE_OUTPUT_MODEL_PATH,
}

missing_files = {name: path for name, path in required_files.items() if not path.exists()}
if missing_files:
    missing_text = "\n".join(f"- {name}: {path}" for name, path in missing_files.items())
    raise FileNotFoundError(f"Missing required preprocessing outputs:\n{missing_text}")

X_train_sequence_raw = np.load(required_files["X_train"]).astype(np.float32)
X_test_sequence_raw = np.load(required_files["X_test"]).astype(np.float32)
y_train_steps_raw = np.load(required_files["y_train_steps"]).astype(np.float32)
y_test_steps_raw = np.load(required_files["y_test_steps"]).astype(np.float32)
y_train_step_mask_raw = np.load(required_files["y_train_step_mask"]).astype(np.float32)
y_test_step_mask_raw = np.load(required_files["y_test_step_mask"]).astype(np.float32)
meta_train_raw = pd.read_csv(required_files["meta_train"])
meta_test_raw = pd.read_csv(required_files["meta_test"])

# 6_modeling.ipynb와 동일하게 full t~t+2 target window만 해석 대상으로 사용합니다.
train_keep = meta_train_raw["target_available_count"].to_numpy() >= TARGET_WINDOW_SIZE
test_keep = meta_test_raw["target_available_count"].to_numpy() >= TARGET_WINDOW_SIZE

X_train_sequence = X_train_sequence_raw[train_keep]
X_test_sequence = X_test_sequence_raw[test_keep]
y_train_steps = y_train_steps_raw[train_keep]
y_test_steps = y_test_steps_raw[test_keep]
y_train_step_mask = y_train_step_mask_raw[train_keep]
y_test_step_mask = y_test_step_mask_raw[test_keep]
meta_train = meta_train_raw.loc[train_keep].reset_index(drop=True)
meta_test = meta_test_raw.loc[test_keep].reset_index(drop=True)

# Single-output LightGBM 모델은 anchor t 시점 feature만 입력으로 사용합니다.
X_train = X_train_sequence[:, -1, :].astype(np.float32)
X_test = X_test_sequence[:, -1, :].astype(np.float32)

if FEATURE_COLUMNS_PATH.exists():
    with open(FEATURE_COLUMNS_PATH, "r", encoding="utf-8") as f:
        feature_columns = json.load(f)
else:
    preprocessor_payload = joblib.load(required_files["preprocessor"])
    feature_columns = preprocessor_payload.get("feature_columns")
    if feature_columns is None:
        raise KeyError(
            f"{FEATURE_COLUMNS_PATH} does not exist and feature_columns was not found in "
            f"{required_files['preprocessor']}"
        )
    print("FEATURE_COLUMNS_PATH not found; loaded feature_columns from lstm_preprocessor.joblib")

if len(feature_columns) != X_train.shape[1]:
    raise ValueError(f"Feature column count mismatch: {len(feature_columns)} names vs {X_train.shape[1]} input features")

print("X_train sequence", X_train_sequence.shape)
print("X_test sequence", X_test_sequence.shape)
print("X_train anchor t", X_train.shape)
print("X_test anchor t", X_test.shape)
print("features", len(feature_columns))


In [ ]:
data_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "source_examples_before_filter": len(meta_train_raw),
            "n_examples": X_train.shape[0],
            "excluded_examples": int((~train_keep).sum()),
            "n_features": X_train.shape[1],
            "masked_positive_rate_within_t_plus_2": float(((y_train_steps * y_train_step_mask).max(axis=1) > 0).mean()),
            "nan_count": int(np.isnan(X_train).sum()),
        },
        {
            "split": "test",
            "source_examples_before_filter": len(meta_test_raw),
            "n_examples": X_test.shape[0],
            "excluded_examples": int((~test_keep).sum()),
            "n_features": X_test.shape[1],
            "masked_positive_rate_within_t_plus_2": float(((y_test_steps * y_test_step_mask).max(axis=1) > 0).mean()),
            "nan_count": int(np.isnan(X_test).sum()),
        },
    ]
)
display(data_summary)


## LightGBM 모델 로딩


In [ ]:
single_model_payload = joblib.load(SINGLE_OUTPUT_MODEL_PATH)
single_output_model = single_model_payload["model"]

print("single-output loaded:", SINGLE_OUTPUT_MODEL_PATH)
print("single-output input:", single_model_payload.get("input"))
print("single-output target:", single_model_payload.get("target"))
print("single-output best_params:", single_model_payload.get("best_params"))


## 예측과 metric helper


In [ ]:
def safe_auprc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    y_true = np.asarray(y_true).astype(int)
    return float(average_precision_score(y_true, y_prob)) if len(np.unique(y_true)) == 2 else np.nan


def safe_auroc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    y_true = np.asarray(y_true).astype(int)
    return float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) == 2 else np.nan


def predict_single_output_proba(model, x: np.ndarray) -> np.ndarray:
    return model.predict_proba(x)[:, 1].astype(float)


def within_t_plus_2_target(y_steps: np.ndarray, y_mask: np.ndarray) -> np.ndarray:
    return ((y_steps * y_mask).max(axis=1) > 0).astype(int)


def single_output_metric_summary(y_true: np.ndarray, y_prob: np.ndarray) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    return {
        "within_t_plus_2_auprc": safe_auprc(y_true, y_prob),
        "within_t_plus_2_auroc": safe_auroc(y_true, y_prob),
        "within_t_plus_2_n": int(len(y_true)),
        "within_t_plus_2_events": int(y_true.sum()),
        "within_t_plus_2_event_rate": float(y_true.mean()) if len(y_true) else np.nan,
    }


def build_single_output_task(y_true: np.ndarray, y_prob: np.ndarray) -> dict:
    return {
        "within_t_plus_2": {
            "label": "within t+2",
            "y_true": np.asarray(y_true).astype(int),
            "y_prob": np.asarray(y_prob).astype(float),
            "active": np.ones(len(y_true), dtype=bool),
        }
    }


In [ ]:
single_y_test = within_t_plus_2_target(y_test_steps, y_test_step_mask)
single_test_prob = predict_single_output_proba(single_output_model, X_test)
single_test_summary = single_output_metric_summary(single_y_test, single_test_prob)
print("Single-output LightGBM within t+2")
display(pd.DataFrame([single_test_summary]))


## Calibration, DCA


In [ ]:
def estimate_calibration_intercept_slope(y_true: np.ndarray, y_prob: np.ndarray) -> tuple[float, float]:
    if len(y_true) == 0 or len(np.unique(y_true)) < 2:
        return np.nan, np.nan

    logit_prob = np.log(y_prob / (1.0 - y_prob)).reshape(-1, 1)
    if np.unique(logit_prob).size < 2:
        return np.nan, np.nan

    try:
        calibration_model = LogisticRegression(penalty=None, solver="lbfgs", max_iter=1000)
        calibration_model.fit(logit_prob, y_true)
    except (TypeError, ValueError):
        try:
            calibration_model = LogisticRegression(penalty="none", solver="lbfgs", max_iter=1000)
            calibration_model.fit(logit_prob, y_true)
        except ValueError:
            return np.nan, np.nan

    return float(calibration_model.intercept_[0]), float(calibration_model.coef_[0, 0])


def expected_calibration_error(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 10) -> float:
    bin_df = pd.DataFrame({"y_true": y_true, "y_prob": y_prob})
    if bin_df.empty:
        return np.nan

    unique_prob_count = bin_df["y_prob"].nunique()
    if unique_prob_count <= 1:
        return float(abs(bin_df["y_true"].mean() - bin_df["y_prob"].mean()))

    bin_count = min(n_bins, unique_prob_count)
    bin_df["bin"] = pd.qcut(bin_df["y_prob"], q=bin_count, duplicates="drop")
    grouped = bin_df.groupby("bin", observed=True)
    weighted_abs_errors = [
        len(group) / len(bin_df) * abs(group["y_true"].mean() - group["y_prob"].mean())
        for _, group in grouped
    ]
    return float(np.sum(weighted_abs_errors))


def calibration_table(tasks: dict, n_bins: int = 10) -> tuple[pd.DataFrame, pd.DataFrame]:
    summary_rows = []
    curve_rows = []

    for task, payload in tasks.items():
        y_true = payload["y_true"]
        y_prob = np.clip(payload["y_prob"], 1e-6, 1 - 1e-6)
        observed = float(np.mean(y_true)) if len(y_true) else np.nan
        predicted = float(np.mean(y_prob)) if len(y_prob) else np.nan
        expected_observed_ratio = observed / predicted if predicted > 0 else np.nan
        calibration_intercept, calibration_slope = estimate_calibration_intercept_slope(y_true, y_prob)
        ece = expected_calibration_error(y_true, y_prob, n_bins=n_bins)
        brier = float(brier_score_loss(y_true, y_prob)) if len(np.unique(y_true)) == 2 else np.nan
        baseline_brier = (
            float(brier_score_loss(y_true, np.full(len(y_true), observed, dtype=float)))
            if len(np.unique(y_true)) == 2
            else np.nan
        )

        summary_rows.append(
            {
                "task": task,
                "label": payload["label"],
                "n": int(len(y_true)),
                "events": int(np.sum(y_true)),
                "event_rate": observed,
                "mean_predicted_probability": predicted,
                "observed_expected_ratio": expected_observed_ratio,
                "calibration_intercept": calibration_intercept,
                "calibration_slope": calibration_slope,
                "expected_calibration_error": ece,
                "brier_score": brier,
                "baseline_brier_score": baseline_brier,
            }
        )

        bin_df = pd.DataFrame({"y_true": y_true, "y_prob": y_prob})
        unique_prob_count = bin_df["y_prob"].nunique()
        if len(bin_df) and unique_prob_count > 1:
            bin_count = min(n_bins, unique_prob_count)
            bin_df["bin"] = pd.qcut(bin_df["y_prob"], q=bin_count, duplicates="drop")
            grouped = bin_df.groupby("bin", observed=True)
            for bin_idx, (_, group) in enumerate(grouped, start=1):
                curve_rows.append(
                    {
                        "task": task,
                        "label": payload["label"],
                        "bin": bin_idx,
                        "n": int(len(group)),
                        "events": int(group["y_true"].sum()),
                        "mean_predicted_probability": float(group["y_prob"].mean()),
                        "observed_event_rate": float(group["y_true"].mean()),
                        "min_predicted_probability": float(group["y_prob"].min()),
                        "max_predicted_probability": float(group["y_prob"].max()),
                    }
                )

    return pd.DataFrame(summary_rows), pd.DataFrame(curve_rows)


def decision_curve(tasks: dict, thresholds: np.ndarray) -> pd.DataFrame:
    rows = []
    for task, payload in tasks.items():
        y_true = payload["y_true"].astype(int)
        y_prob = payload["y_prob"].astype(float)
        n = len(y_true)
        prevalence = float(np.mean(y_true)) if n else np.nan

        for threshold in thresholds:
            if threshold <= 0 or threshold >= 1 or n == 0:
                continue
            y_pred = y_prob >= threshold
            tp = int(np.sum(y_pred & (y_true == 1)))
            fp = int(np.sum(y_pred & (y_true == 0)))
            odds = threshold / (1.0 - threshold)
            model_net_benefit = (tp / n) - (fp / n) * odds
            treat_all_net_benefit = prevalence - (1.0 - prevalence) * odds
            rows.extend(
                [
                    {
                        "task": task,
                        "label": payload["label"],
                        "threshold": float(threshold),
                        "strategy": "model",
                        "net_benefit": float(model_net_benefit),
                        "standardized_net_benefit": float(model_net_benefit / prevalence) if prevalence > 0 else np.nan,
                    },
                    {
                        "task": task,
                        "label": payload["label"],
                        "threshold": float(threshold),
                        "strategy": "treat_all",
                        "net_benefit": float(treat_all_net_benefit),
                        "standardized_net_benefit": float(treat_all_net_benefit / prevalence) if prevalence > 0 else np.nan,
                    },
                    {
                        "task": task,
                        "label": payload["label"],
                        "threshold": float(threshold),
                        "strategy": "treat_none",
                        "net_benefit": 0.0,
                        "standardized_net_benefit": 0.0,
                    },
                ]
            )
    return pd.DataFrame(rows)


In [ ]:
single_output_tasks = build_single_output_task(single_y_test, single_test_prob)
single_calibration_summary, single_calibration_curve = calibration_table(single_output_tasks, n_bins=CALIBRATION_N_BINS)
single_dca_results = decision_curve(single_output_tasks, thresholds=DCA_THRESHOLDS)

single_calibration_summary.to_csv(OUTPUT_DIR / "lgbm_single_output_calibration_summary.csv", index=False)
single_calibration_curve.to_csv(OUTPUT_DIR / "lgbm_single_output_calibration_curve.csv", index=False)
single_dca_results.to_csv(OUTPUT_DIR / "lgbm_single_output_decision_curve.csv", index=False)

print("Single-output LightGBM calibration summary")
display(single_calibration_summary)


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
calibration_text = None
for task, payload in single_output_tasks.items():
    curve = single_calibration_curve[single_calibration_curve["task"] == task]
    if curve.empty:
        continue
    summary_row = single_calibration_summary[single_calibration_summary["task"] == task]
    if not summary_row.empty:
        ece = summary_row["expected_calibration_error"].iloc[0]
        brier = summary_row["brier_score"].iloc[0]
        if "baseline_brier_score" in summary_row.columns:
            baseline_brier = summary_row["baseline_brier_score"].iloc[0]
        else:
            y_true = payload["y_true"]
            observed = float(np.mean(y_true)) if len(y_true) else np.nan
            baseline_brier = (
                float(brier_score_loss(y_true, np.full(len(y_true), observed, dtype=float)))
                if len(np.unique(y_true)) == 2
                else np.nan
            )
        calibration_text = f"ECE={ece:.3f}\nBrier={brier:.3f} (Baseline: {baseline_brier:.3f})"
    ax.plot(
        curve["mean_predicted_probability"],
        curve["observed_event_rate"],
        marker="o",
        linewidth=1.8,
    )
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
ax.set_title("Single-output LightGBM calibration")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed event rate")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
if calibration_text is not None:
    ax.text(0.04, 0.96, calibration_text, transform=ax.transAxes, ha="left", va="top", fontsize=10)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "lgbm_single_output_calibration.png", dpi=200, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(6.5, 5.5))
for task, payload in single_output_tasks.items():
    model_curve = single_dca_results[(single_dca_results["task"] == task) & (single_dca_results["strategy"] == "model")]
    if model_curve.empty:
        continue
    ax.plot(model_curve["threshold"], model_curve["net_benefit"], linewidth=1.8, label=payload["label"])
within_all_curve = single_dca_results[
    (single_dca_results["task"] == "within_t_plus_2") & (single_dca_results["strategy"] == "treat_all")
]
within_none_curve = single_dca_results[
    (single_dca_results["task"] == "within_t_plus_2") & (single_dca_results["strategy"] == "treat_none")
]
ax.plot(within_all_curve["threshold"], within_all_curve["net_benefit"], linestyle="--", color="gray", linewidth=1, label="treat all")
ax.plot(within_none_curve["threshold"], within_none_curve["net_benefit"], linestyle=":", color="black", linewidth=1, label="treat none")
ax.set_title("Single-output LightGBM decision curve analysis")
ax.set_xlabel("Threshold probability")
ax.set_ylabel("Net benefit")
ax.axhline(0, color="black", linewidth=0.8, alpha=0.5)
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "lgbm_single_output_decision_curve.png", dpi=200, bbox_inches="tight")
plt.show()


## 해석 대상 subset 선택


In [ ]:
X_explain_source = X_test
y_explain_steps = y_test_steps
y_explain_mask = y_test_step_mask
meta_explain = meta_test

n_explain = min(EXPLAIN_SAMPLE_SIZE, len(X_explain_source))
explain_idx = rng.choice(len(X_explain_source), size=n_explain, replace=False)
X_explain = X_explain_source[explain_idx].copy()
y_explain_steps_sub = y_explain_steps[explain_idx].copy()
y_explain_mask_sub = y_explain_mask[explain_idx].copy()
meta_explain_sub = meta_explain.iloc[explain_idx].reset_index(drop=True)

single_y_explain = within_t_plus_2_target(y_explain_steps_sub, y_explain_mask_sub)
single_baseline_prob = predict_single_output_proba(single_output_model, X_explain)
single_baseline_metrics = single_output_metric_summary(single_y_explain, single_baseline_prob)
print("single-output explain subset:", X_explain.shape)
display(pd.DataFrame([single_baseline_metrics]))


## Permutation feature importance


In [ ]:
# Single-output LightGBM permutation feature importance
def single_output_permutation_feature_importance(
    model,
    x: np.ndarray,
    y_true: np.ndarray,
    feature_names: list[str],
    repeats: int,
) -> pd.DataFrame:
    baseline_prob = predict_single_output_proba(model, x)
    baseline = single_output_metric_summary(y_true, baseline_prob)
    feature_indices = list(range(x.shape[1]))

    rows = []
    for feature_idx in feature_indices:
        repeat_rows = []
        for repeat in range(1, repeats + 1):
            x_perm = x.copy()
            perm = rng.permutation(x_perm.shape[0])
            x_perm[:, feature_idx] = x_perm[perm, feature_idx]
            perm_prob = predict_single_output_proba(model, x_perm)
            repeat_rows.append(single_output_metric_summary(y_true, perm_prob))

        repeat_df = pd.DataFrame(repeat_rows)
        row = {
            "feature_idx": feature_idx,
            "feature": feature_names[feature_idx],
            "repeats": repeats,
        }
        for metric_name, baseline_value in baseline.items():
            if metric_name.endswith("_n") or metric_name.endswith("_events") or metric_name.endswith("_event_rate"):
                continue
            perm_mean = float(repeat_df[metric_name].mean())
            perm_std = float(repeat_df[metric_name].std(ddof=0))
            row[f"baseline_{metric_name}"] = baseline_value
            row[f"permuted_mean_{metric_name}"] = perm_mean
            row[f"permuted_std_{metric_name}"] = perm_std
            row[f"drop_{metric_name}"] = baseline_value - perm_mean
        rows.append(row)

    return pd.DataFrame(rows)


single_feature_importance = single_output_permutation_feature_importance(
    model=single_output_model,
    x=X_explain,
    y_true=single_y_explain,
    feature_names=feature_columns,
    repeats=PERMUTATION_REPEATS,
)
single_feature_importance = single_feature_importance.sort_values("drop_within_t_plus_2_auprc", ascending=False).reset_index(drop=True)
single_feature_importance.to_csv(OUTPUT_DIR / "lgbm_single_output_permutation_feature_importance.csv", index=False)
display(single_feature_importance.head(30))


In [ ]:
plot_df = single_feature_importance.head(TOP_N_PLOT).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, max(5, 0.28 * len(plot_df))))
ax.barh(plot_df["feature"], plot_df["drop_within_t_plus_2_auprc"], color="tab:blue")
ax.set_xlabel("AUPRC drop after permutation")
ax.set_ylabel("Feature")
ax.set_title("LightGBM single-output permutation feature importance")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "lgbm_single_output_permutation_feature_importance_top.png", dpi=200, bbox_inches="tight")
plt.show()


## LightGBM built-in gain importance


In [ ]:
# Single-output LightGBM built-in gain importance
single_booster = getattr(single_output_model, "booster_", None)
if single_booster is None:
    raise AttributeError("Single-output LightGBM estimator does not have booster_ attribute. Was it fitted?")

single_gain_values = single_booster.feature_importance(importance_type="gain")
single_split_values = single_booster.feature_importance(importance_type="split")
single_gain_importance = pd.DataFrame(
    {
        "feature_idx": np.arange(len(feature_columns)),
        "feature": feature_columns,
        "gain": single_gain_values.astype(float),
        "split": single_split_values.astype(int),
    }
)
single_gain_importance["gain_rank"] = single_gain_importance["gain"].rank(ascending=False, method="first").astype(int)
single_gain_importance = single_gain_importance.sort_values("gain_rank")
single_gain_importance.to_csv(OUTPUT_DIR / "lgbm_single_output_gain_feature_importance.csv", index=False)
display(single_gain_importance.head(15))


In [ ]:
plot_df = single_gain_importance.head(TOP_N_PLOT).sort_values("gain").copy()
fig, ax = plt.subplots(figsize=(8, max(5, 0.28 * len(plot_df))))
ax.barh(plot_df["feature"], plot_df["gain"], color="tab:blue")
ax.set_xlabel("Gain")
ax.set_ylabel("Feature")
ax.set_title("LightGBM single-output gain importance")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "lgbm_single_output_gain_importance_top.png", dpi=200, bbox_inches="tight")
plt.show()


## SHAP


In [ ]:
print("RUN_SHAP:", RUN_SHAP)

if RUN_SHAP:
    import shap

    single_ex_n = min(SHAP_EXPLAIN_SIZE, len(X_explain))
    single_shap_x = X_explain[:single_ex_n]
    single_explainer = shap.TreeExplainer(single_output_model)
    single_raw_shap_values = single_explainer.shap_values(single_shap_x)
    if isinstance(single_raw_shap_values, list):
        single_shap_values = np.asarray(single_raw_shap_values[-1])
    else:
        single_shap_values = np.asarray(single_raw_shap_values)
        if single_shap_values.ndim == 3 and single_shap_values.shape[-1] == 2:
            single_shap_values = single_shap_values[:, :, -1]
        elif single_shap_values.ndim == 3 and single_shap_values.shape[0] == 2:
            single_shap_values = single_shap_values[-1]

    if single_shap_values.ndim != 2:
        raise ValueError(f"Unexpected SHAP value shape for single-output LightGBM binary classifier: {single_shap_values.shape}")

    single_mean_abs_by_feature = np.abs(single_shap_values).mean(axis=0)
    single_shap_feature_importance = pd.DataFrame(
        {
            "feature_idx": np.arange(len(feature_columns)),
            "feature": feature_columns,
            "mean_abs_shap": single_mean_abs_by_feature,
            "target": "within_t_plus_2",
        }
    ).sort_values("mean_abs_shap", ascending=False)
    single_shap_feature_importance.to_csv(OUTPUT_DIR / "lgbm_single_output_shap_feature_importance.csv", index=False)
    display(single_shap_feature_importance.head(30))

    plot_df = single_shap_feature_importance.head(TOP_N_PLOT).iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, max(5, 0.28 * len(plot_df))))
    ax.barh(plot_df["feature"], plot_df["mean_abs_shap"], color="tab:blue")
    ax.set_xlabel("Mean |SHAP value|")
    ax.set_ylabel("Feature")
    ax.set_title("LightGBM single-output SHAP feature importance")
    ax.grid(axis="x", alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "lgbm_single_output_shap_feature_importance_top.png", dpi=200, bbox_inches="tight")
    plt.show()

    shap.summary_plot(
        single_shap_values,
        pd.DataFrame(single_shap_x, columns=feature_columns),
        max_display=TOP_N_PLOT,
        show=False,
    )
    plt.title("LightGBM single-output SHAP beeswarm")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "lgbm_single_output_shap_beeswarm.png", dpi=200, bbox_inches="tight")
    plt.show()


## 저장된 산출물


In [ ]:
saved_files = sorted([
    str(path.relative_to(PROJECT_DIR))
    for path in OUTPUT_DIR.glob("lgbm_single_output*.csv")
])
saved_figures = sorted([
    str(path.relative_to(PROJECT_DIR))
    for path in FIGURE_DIR.glob("lgbm_single_output*.png")
])
print("LightGBM CSV outputs:")
for path in saved_files:
    print("-", path)
print("LightGBM figure outputs:")
for path in saved_figures:
    print("-", path)
